# Question 5: Full RAG with OpenAI

Rebuild the book index, retrieve five chunks, and answer `python function definition` using the homework's supplied instructions and prompt template. Return the answer and the API's reported input/output token counts in one call.

Run the cells in order with the project's Python environment. The OpenAI client reads `OPENAI_API_KEY` from your environment or the repository's `.env` file. The final cell makes a paid OpenAI API request using `gpt-4o-mini`.

In [ ]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# Support running from the repository root or homework/.
project_root = Path.cwd()
if not (project_root / "books_text").is_dir():
    project_root = project_root.parent

books_text_dir = project_root / "books_text"
markdown_files = sorted(books_text_dir.glob("*.md"))
if not markdown_files:
    raise FileNotFoundError(f"No Markdown books found in {books_text_dir}")

## Load and chunk all books

Keep non-empty lines and apply the same 100-line window and 50-line step as Question 2.

In [ ]:
books = []
for markdown_path in markdown_files:
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    books.append({
        "source": markdown_path.name,
        "content": [line for line in lines if line.strip()],
    })

chunks = chunk_documents(books, size=100, step=50)
print(f"Created {len(chunks):,} chunks from {len(books)} books.")

## Prepare searchable documents

MinSearch expects text strings. Join each chunk’s lines with newlines, preserving its source and starting position without modifying the original chunk.

In [ ]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents


documents = prepare_documents(chunks)

## Build the search index

In [ ]:
index = Index(text_fields=["content"])
index.fit(documents)

print(f"Number of documents (chunks) indexed: {len(documents)}")

## Initialize the OpenAI client

Load the repository’s `.env` without overriding existing environment variables.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(project_root / ".env", override=False)
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or the repository's .env file.")

openai_client = OpenAI()

## Define the RAG functions

Keep the homework’s instructions, JSON context, prompt template, five search results, and default model. Extend the return values to include actual token usage reported by OpenAI.

In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## Run the query and report the homework answers

This cell sends one API request. The input count includes the system instructions, question, JSON context, and message formatting. Re-running it makes another request; generated answers and output counts can vary.

In [ ]:
result = rag("python function definition")

print("Response:\n")
print(result["answer"])
print(f"\nInput tokens: {result['input_tokens']}")
print(f"Output tokens: {result['output_tokens']}")